In [ ]:
!wget https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv

In [3]:
import csv
import requests
import os


In [ ]:

os.makedirs('books', exist_ok=True)

with open('books.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        url = row['pdf_url']
        file_name = url.split('/')[-1]
        response = requests.get(url)
        with open(f'books/{file_name}', 'wb') as pdf_file:
            pdf_file.write(response.content)
        print(f"Downloaded: {file_name}")

In [1]:
!uv add 'markitdown[pdf]'

Using CPython 3.13.13 interpreter at: /usr/local/bin/python3.13
Removed virtual environment at: /workspaces/ai-engineering-buildcamp/.venv
Creating virtual environment at: /workspaces/ai-engineering-buildcamp/.venv
Resolved 139 packages in 1ms
⠹ Preparing packages... (0/134)                                                 
⠹ Preparing packages... (0/134)-----------------     0 B/15.86 MiB           
⠹ Preparing packages... (0/134)-----------------     0 B/15.86 MiB           
magika               ------------------------------     0 B/14.66 MiB
⠹ Preparing packages... (0/134)-----------------     0 B/15.86 MiB           
magika               ------------------------------     0 B/14.66 MiB
⠹ Preparing packages... (0/134)-----------------     0 B/15.86 MiB           
magika               ------------------------------     0 B/14.66 MiB
⠹ Preparing packages... (0/134)-----------------     0 B/15.86 MiB           
notebook             ------------------------------     0 B/13.90 MiB
magik

In [4]:
from markitdown import MarkItDown

os.makedirs('books_text', exist_ok=True)
md = MarkItDown()

for file in os.listdir('books'):
    if file.endswith('.pdf'):
        pdf_path = os.path.join('books', file)
        text_path = os.path.join('books_text', file.replace('.pdf', '.md'))
        
        result = md.convert(pdf_path)
        
        with open(text_path, 'w', encoding='utf-8') as f:
            f.write(result.text_content)
        
        print(f"Converted: {file}")

Converted: thinkjava2.pdf
Converted: PhysicalModelingInMatlab4.pdf
Converted: Think-C.pdf
Converted: thinkos.pdf
Converted: thinkcomplexity2.pdf
Converted: thinkpython2.pdf
Converted: thinkdsp.pdf


In [5]:
!wc -l books_text/thinkpython2.md

16268 books_text/thinkpython2.md


In [6]:
documents = []
for file in os.listdir('books_text'):
    if file.endswith('.md'):
        with open(f'books_text/{file}', 'r') as f:
            lines = [line.strip() for line in f if line.strip()]
        documents.append({
            'source': file,
            'content': lines
        })

In [7]:
!uv add gitsource

Resolved 139 packages in 46ms
Checked 134 packages in 68ms


In [8]:
from gitsource import chunk_documents
chunked_docs = chunk_documents(documents=documents, size=100, step=50)

In [11]:
think_python_chunks = [c for c in chunked_docs if 'thinkpython2' in c['source']]
print(len(think_python_chunks))

214


In [12]:
!uv add minsearch

Resolved 139 packages in 2ms
Checked 134 packages in 1ms


In [13]:
from minsearch import Index

for chunk in chunked_docs:
    chunk['content'] = '\n'.join(chunk['content'])

index = Index(
    text_fields=['content'],
    keyword_fields=['source']
)
index.fit(chunked_docs)
print(len(chunked_docs))

1009


In [14]:
print(len(chunked_docs))
for doc in documents:
    book_chunks = [c for c in chunked_docs if c['source'] == doc['source']]
    print(f"{doc['source']}: {len(book_chunks)} chunks")

1009
Think-C.md: 107 chunks
thinkos.md: 68 chunks
thinkcomplexity2.md: 144 chunks
thinkjava2.md: 247 chunks
thinkpython2.md: 214 chunks
thinkdsp.md: 110 chunks
PhysicalModelingInMatlab4.md: 119 chunks


In [15]:
results = index.search("python function definition", num_results=5)
print(results[0]['source'])

thinkpython2.md


In [16]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [19]:
import json

from prompt_toolkit import prompt

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions=None, model="llama-3.3-70b-versatile"):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content, response.usage.prompt_tokens, response.usage.completion_tokens

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    print(f"Input tokens: {input_tokens}, Output tokens: {output_tokens}")
    return answer


In [20]:
rag('python function definition')

Input tokens: 7236, Output tokens: 359


'In Python, a function definition is a statement that creates a new function, specifying its name, parameters, and the statements it contains. The general syntax of a function definition in Python is:\n\n```\ndef function_name(parameters):\n    # function body\n```\n\nFor example, a simple function that takes no parameters and prints "Hello, World!" would be defined like this:\n\n```\ndef hello():\n    print(\'Hello, World!\')\n```\n\nYou can also define functions with parameters. For example, a function that takes a name as a parameter and prints a greeting would be defined like this:\n\n```\ndef greet(name):\n    print(\'Hello, \' + name + \'!\')\n```\n\nYou can call this function by passing a name as an argument, like this:\n\n```\ngreet(\'John\')\n```\n\nThis would print "Hello, John!".\n\nFunctions can also return values. For example, a function that takes two numbers as parameters and returns their sum would be defined like this:\n\n```\ndef add(a, b):\n    return a + b\n```\n\nY

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


In [ ]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def llm_structured(user_prompt, output_type, instructions=None, model="llama-3.3-70b-versatile"):
    schema = output_type.model_json_schema()
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions + f"\n\nYour response must be a JSON object with the following structure:\n{schema}\n\nReturn only the JSON object with actual values, not the schema itself."
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content.strip()
    # print(raw)
    event = output_type(**json.loads(raw))
    return(event),response.usage.prompt_tokens, response.usage.completion_tokens

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm_structured(prompt, RAGResponse, instructions)
    print(f"Input tokens: {input_tokens}, Output tokens: {output_tokens}")
    return answer


In [23]:
rag('python function definition')

{
  "answer": "In Python, a function definition is a statement that creates a new function, specifying its name, parameters, and the statements it contains. The general syntax of a function definition in Python is `def function_name(parameters):` followed by the statements that make up the function body.",
   "found_answer": true,
   "confidence": 0.8,
   "confidence_explanation": "The provided context contains information about functions in Python and other programming languages, which allows for a confident answer.",
   "answer_type": "explanation",
   "followup_questions": [
       "What is the difference between a function and a method in Python?",
       "How do you call a function in Python?",
       "What are the different types of functions in Python?"
   ]
}
Input tokens: 7535, Output tokens: 162


RAGResponse(answer='In Python, a function definition is a statement that creates a new function, specifying its name, parameters, and the statements it contains. The general syntax of a function definition in Python is `def function_name(parameters):` followed by the statements that make up the function body.', found_answer=True, confidence=0.8, confidence_explanation='The provided context contains information about functions in Python and other programming languages, which allows for a confident answer.', answer_type='explanation', followup_questions=['What is the difference between a function and a method in Python?', 'How do you call a function in Python?', 'What are the different types of functions in Python?'])